# FHIR resource mapping experiment

Runs one experiment from `references/configs/config.yaml` with the same code as `scripts/run_experiment.py` (`src/pipeline.py`). Paths in the config are relative to the repository root, so the notebook can be run from any directory.

MLflow logging is enabled when `MLFLOW_TRACKING_URI` is set (e.g. `http://localhost:8080` after `make start_tracking`); `HF_KEY` (only needed for gated models) can be put in `notebooks/.env`.

In [ ]:
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

ROOT = Path.cwd().resolve()
while not (ROOT / "src" / "pipeline.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
load_dotenv(".env")

from src.pipeline import ExperimentRunner, load_config

In [ ]:
# config
cfg = load_config(ROOT / "references" / "configs" / "config.yaml")
cfg

In [ ]:
# data → vectorstore → inference (CodeCarbon) → metrics → outputs/ and MLflow
runner = ExperimentRunner(
    output_dir   = ROOT / "outputs",
    tracking_uri = os.environ.get("MLFLOW_TRACKING_URI"),
)
summary = runner.run(cfg)

In [ ]:
# metrics
print(summary["run_name"], "| failed:", summary["n_failed"], "| co2_kg:", round(summary["co2_kg"], 4))
summary["metrics"]